# Headline Mood Dashboard - Colab Demo

This notebook demonstrates the same functionality as the Hugging Face Space.
Paste 3-10 news headlines and see their aggregate and individual emotional profiles.

**Model:** `j-hartmann/emotion-english-distilroberta-base` (7 emotions)

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline
from IPython.display import display, HTML

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)
print("Model loaded!")

In [ ]:
EMOTIONS = {
    "anger":    {"color": "#e74c3c", "emoji": "\U0001f620"},
    "disgust":  {"color": "#27ae60", "emoji": "\U0001f922"},
    "fear":     {"color": "#8e44ad", "emoji": "\U0001f628"},
    "joy":      {"color": "#f1c40f", "emoji": "\U0001f60a"},
    "neutral":  {"color": "#95a5a6", "emoji": "\U0001f610"},
    "sadness":  {"color": "#3498db", "emoji": "\U0001f622"},
    "surprise": {"color": "#e67e22", "emoji": "\U0001f632"},
}

## Analyze Headlines

Edit the headlines list below, then run the cell.

In [ ]:
headlines = [
    "Scientists discover new species of whale in Pacific Ocean",
    "Stock market plunges amid global uncertainty",
    "Local hero saves child from burning building",
    "New study reveals alarming pollution levels",
    "Tech giant announces revolutionary AI breakthrough",
]

# Analyze each headline
all_results = []
for headline in headlines:
    scores = classifier(headline)[0]
    score_map = {r["label"]: r["score"] for r in scores}
    top_e = max(score_map, key=score_map.get)
    all_results.append({"headline": headline, "scores": score_map, "top": top_e})
    print(f"[{top_e}] {headline}")

print("\nDone!")

## Aggregate Mood Profile

In [ ]:
# Aggregate scores
agg = {e: 0 for e in EMOTIONS}
for r in all_results:
    for e in EMOTIONS:
        agg[e] += r["scores"].get(e, 0)
total = sum(agg.values())

# Legend
legend = ' '.join(
    f'<span style="display:inline-flex;align-items:center;gap:4px;margin:0 6px;">'
    f'<span style="width:12px;height:12px;border-radius:3px;background:{EMOTIONS[e]["color"]};display:inline-block;"></span>'
    f'<span style="font-size:0.82em;">{e}</span></span>'
    for e in EMOTIONS
)

# Aggregate stacked bar
agg_segs = ''
for e in EMOTIONS:
    pct = (agg[e] / total * 100) if total > 0 else 0
    agg_segs += f'<div style="width:{pct}%;background:{EMOTIONS[e]["color"]};height:100%;min-width:1px;" title="{e}: {pct:.0f}%"></div>'

# Individual headline cards
cards = ''
for r in all_results:
    info = EMOTIONS.get(r["top"], {"color":"#999","emoji":"?"})
    h_total = sum(r["scores"].values())
    mini_segs = ''
    for e in EMOTIONS:
        pct = (r["scores"].get(e, 0) / h_total * 100) if h_total > 0 else 0
        mini_segs += f'<div style="width:{pct}%;background:{EMOTIONS[e]["color"]};height:100%;"></div>'

    cards += f'''
    <div style="padding:10px;border:1px solid #eee;border-radius:8px;margin:6px 0;background:#fafafa;">
        <div style="display:flex;justify-content:space-between;align-items:center;">
            <span style="font-size:0.9em;">{r["headline"][:80]}</span>
            <span style="background:{info["color"]};color:white;padding:2px 8px;border-radius:10px;font-size:0.75em;">{r["top"]}</span>
        </div>
        <div style="display:flex;height:8px;border-radius:4px;overflow:hidden;margin-top:8px;">{mini_segs}</div>
    </div>'''

display(HTML(f'''
<div style="font-family:system-ui;max-width:700px;">
    <div style="margin-bottom:12px;">{legend}</div>
    <h4 style="margin:8px 0 4px;">Overall Mood</h4>
    <div style="display:flex;height:24px;border-radius:6px;overflow:hidden;border:1px solid #ddd;">{agg_segs}</div>
    <h4 style="margin:16px 0 8px;">Individual Headlines</h4>
    {cards}
</div>
'''))

## Try Your Own Headlines

Edit the `headlines` list in the cell above and re-run both cells to see different results.

Ideas:
- Copy real headlines from a news site
- Compare headlines from different news sources covering the same story
- Write your own headlines and see what emotions the model detects